<a href="https://colab.research.google.com/github/mateoneo/CatorDog/blob/main/CatorDogclassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount = True)

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive"

'130B Homework 1.gdoc'
'20220601_215903 (1).mp4'
 20220601_215903.mp4
'2022-07-18 (6).png'
'2022-07-18 (7).png'
'2.11 Language development..gdoc'
'2.1 Notes Geometry.gdoc'
'2.1 one notes.gdoc'
'2.23 language dev.gdoc'
'2.4 notes.gdoc'
'2.7 1 and 2.gdoc'
'2.7 1 video.gdoc'
'3.25 language development.gdoc'
'3.4 vid.gdoc'
'3.6 vid.gdoc'
'3rd law corrections.gdoc'
'4.13 language development. .gdoc'
'4.2 vids 1-3.gdoc'
'4.3 vid 3 and 4.gdoc'
'4.3 vid 7 and 8.gdoc'
'4.3 vids 1-2.gdoc'
'4.6 youtube vid note.gdoc'
'4.8 khan academy video .gdoc'
'50 word final outliers..gdoc'
'50 word summary ch 6-8 .gdoc'
'50 word summary.gdoc'
'5.20 Cell division mateo cole.gdoc'
'5.2 vids 123.gdoc'
 5.31.gdoc
'5.4 vids 4 and 5.gdoc'
'5.4 vids 6 and 7.gdoc'
 5.gdoc
'5 para essay mateo cole.gdoc'
'7.1 1-3.gdoc'
'7.2 1-6.gdoc'
'8 key issue 3.gdoc'
' A2C5A85 Phil001 paper 2.gdoc'
' A2C5A85 Phil001 paper.gdoc'
'Adultification article notes.gdoc'
'Adultification article notes.pdf'
'AF and HB.gdoc'
'ah flipping .gs

In [ ]:
!cp -r "/content/drive/MyDrive/catdog" /content/data
!ls /content/data

c1.jpg	c2.jpg	c3.jpg	c4.jpg	catdog	d1.jpg	d2.jpg	d3.jpg	d4.jpg


In [ ]:
import glob
print(len(glob.glob('/content/data/**/*.jpg', recursive=True)), "jpgs found")

20 jpgs found


In [ ]:
import os, shutil, glob
os.makedirs('/content/sorted/cats', exist_ok=True)
os.makedirs('/content/sorted/dogs', exist_ok=True)
for f in glob.glob('/content/data/**/*.jpg', recursive=True):
    name = os.path.basename(f).lower()
    if 'cat' in name:
        shutil.copy(f, '/content/sorted/cats')
    elif 'dog' in name:
        shutil.copy(f, '/content/sorted/dogs')
print(len(os.listdir('/content/sorted/cats')), "cats,", len(os.listdir('/content/sorted/dogs')), "dogs")

0 cats, 0 dogs


In [ ]:
import os, shutil, glob
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import torch
import torch.nn.functional as F
import torch.optim as optim

# Correctly sort the files since the filenames are 'c1.jpg', 'd1.jpg', etc.
os.makedirs('/content/sorted/cats', exist_ok=True)
os.makedirs('/content/sorted/dogs', exist_ok=True)

for f in glob.glob('/content/data/**/*.jpg', recursive=True):
    name = os.path.basename(f).lower()
    if name.startswith('c'):
        shutil.copy(f, '/content/sorted/cats')
    elif name.startswith('d'):
        shutil.copy(f, '/content/sorted/dogs')

print(f"Sorted files successfully: {len(os.listdir('/content/sorted/cats'))} cats, {len(os.listdir('/content/sorted/dogs'))} dogs")

# Define transforms and Dataset
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

data = datasets.ImageFolder('/content/sorted', transform=transform)
print("Dataset classes:", data.classes)
trainloader = DataLoader(data, batch_size=32, shuffle=True)
train_size = int(0.8 * len(data))
train_data, val_data = random_split(data, [train_size, len(data) - train_size])
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


class Net(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(3, 32, 3, padding=1)
        self.pool = torch.nn.MaxPool2d(2, 2, padding=1)
        self.conv2 = torch.nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = torch.nn.Conv2d(64, 128, 3, padding=1)
        # For 128x128 input, the shape after 3 poolings becomes 17x17
        self.fc1 = torch.nn.Linear(128 * 17 * 17, 2)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        return x

net = Net()
criterion = torch.nn.CrossEntropyLoss()
learningrate = 0.001
device = torch.device("cpu")
optimizer = torch.optim.Adam(net.parameters(), lr=learningrate)
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            if outputs.ndim == 1 or outputs.shape[1] == 1:   # one output neuron (sigmoid)
                preds = (torch.sigmoid(outputs.view(-1)) > 0.5).long()
            else:                                            # two output neurons (softmax)
                preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.tolist())
    return all_labels, all_preds

for epoch in range(30):
  running_loss = 0.0
  if epoch == 15:
    learningrate = 0.001
  for i, data in enumerate(train_loader, 0):
    inputs, labels = data
    optimizer.zero_grad()
    outputs = net(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item()
    if i % 10 == 9 or i == len(trainloader) - 1:
      print('[%d, %5d] loss: %.3f' % (epoch + 1, i + 1, running_loss / (i + 1)))
      running_loss = 0.0
labels, preds = evaluate(net, val_loader, device)

print("Accuracy:", round(accuracy_score(labels, preds), 4))
print("\nConfusion matrix (rows = actual, cols = predicted):")
print(confusion_matrix(labels, preds))
print("\n", classification_report(labels, preds, target_names=["cat", "dog"]))

Sorted files successfully: 6 cats, 6 dogs
Dataset classes: ['cats', 'dogs']
[1,     1] loss: 0.690
[2,     1] loss: 0.980
[3,     1] loss: 0.647
[4,     1] loss: 0.691
[5,     1] loss: 0.677
[6,     1] loss: 0.638
[7,     1] loss: 0.602
[8,     1] loss: 0.578
[9,     1] loss: 0.534
[10,     1] loss: 0.493
[11,     1] loss: 0.444
[12,     1] loss: 0.383
[13,     1] loss: 0.327
[14,     1] loss: 0.267
[15,     1] loss: 0.206
[16,     1] loss: 0.170
[17,     1] loss: 0.155
[18,     1] loss: 0.089
[19,     1] loss: 0.077
[20,     1] loss: 0.049
[21,     1] loss: 0.037
[22,     1] loss: 0.019
[23,     1] loss: 0.020
[24,     1] loss: 0.008
[25,     1] loss: 0.004
[26,     1] loss: 0.003
[27,     1] loss: 0.003
[28,     1] loss: 0.002
[29,     1] loss: 0.001
[30,     1] loss: 0.000
Accuracy: 0.6667

Confusion matrix (rows = actual, cols = predicted):
[[2 0]
 [1 0]]

               precision    recall  f1-score   support

         cat       0.67      1.00      0.80         2
         dog     

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
